# تطبيق K-Means لليوم الثاني — التعلم غير الخاضع للإشراف
كل خلية تجيب عن سؤال واحد. لا توجد أكواد ناقصة.

## 0) تحميل البيانات

### 0.1 استيراد pandas

In [ ]:
import pandas as pd

### 0.2 قراءة الملف

In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/hanenalmayouf/FAML/main/public/data/manafeth_customers.csv'
df = pd.read_csv(DATA_URL)
print('تم التحميل ✅', df.shape)

## 1) اختيار الخصائص

### 1.1 تحديد أسماء الخصائص السلوكية

In [ ]:
features = [
    'orders_per_month',
    'avg_basket_sar',
    'days_since_last_order',
    'promo_usage_rate'
]

### 1.2 إنشاء X من دون هدف y

In [ ]:
X_cluster = df[features].copy()

### 1.3 مشاهدة ملخص الخصائص

In [ ]:
display(X_cluster.describe().round(2))

## 2) تجهيز الأرقام

### 2.1 تعويض المفقود بالوسيط

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')
X_filled = imputer.fit_transform(X_cluster)

### 2.2 تطبيق التحجيم (Scaling)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_filled)

**لماذا؟** K-Means تعتمد على المسافة، لذلك نوحّد المقاييس قبل التجميع.

## 3) مقارنة قيم K

### 3.1 استيراد الأدوات

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

### 3.2 تجربة K من 2 إلى 6
هذه الخلية تكرر **الفكرة نفسها** لخمس قيم، لذلك وجود الحلقة هنا مقصود.

In [ ]:
k_results = []
for k in range(2, 7):
    candidate = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = candidate.fit_predict(X_scaled)
    k_results.append({
        'K': k,
        'Silhouette': silhouette_score(X_scaled, labels)
    })

### 3.3 عرض جدول المقارنة

In [ ]:
k_table = pd.DataFrame(k_results)
display(k_table.round(3))

## 4) بناء المثال النهائي باستخدام K = 3

### 4.1 إنشاء النموذج

In [ ]:
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)

### 4.2 اكتشاف المجموعات

In [ ]:
cluster_labels = kmeans.fit_predict(X_scaled)

### 4.3 إضافة رقم المجموعة

In [ ]:
clustered_df = df.copy()
clustered_df['cluster'] = cluster_labels

### 4.4 حساب متوسط كل خاصية داخل كل مجموعة

In [ ]:
cluster_profile = (
    clustered_df.groupby('cluster')[features]
    .mean()
    .round(2)
)
display(cluster_profile)

### 4.5 معرفة الخاصية الأكثر اختلافًا بين المجموعات

In [ ]:
feature_range = cluster_profile.max() - cluster_profile.min()
strongest_feature = feature_range.idxmax()
print('أوضح خاصية تفصل المجموعات:', strongest_feature)

## الخلاصة
اخترنا الخصائص، عوضنا المفقود، طبقنا Scaling، قارنا قيم K، ثم فسرنا المجموعات من متوسطاتها.